# DreamerV2 / V3 — Minecraft & Atari Evaluation

**Kernel:** `dreamerv3` conda env (`/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3`)

| Section | Content |
|---------|----------|
| A0 | Import `seollab` + optional HF login (`HF_TOKEN` env) |
| A1 | Paths & JAX |
| A2 | Minecraft V3 env rollout GIF (local trained ckpt) |
| A3 | Local Minecraft score distribution |
| A4 | Minecraft baselines plot (official V3 vs PPO/IMPALA) |
| A5 | Atari V2 vs V3 curves + HNS distribution |
| A6 | Report |

**Checkpoints:** [HyunseoYun/dreamerv3-custom-envs](https://huggingface.co/HyunseoYun/dreamerv3-custom-envs) (Highway). Minecraft uses `vendor/dreamerv3/logdir/minecraft_diamond_full`. Atari V2/V3 comparison uses official published scores (DreamerV2 has no JAX Minecraft support).

---
# Highway Roundabout Inference (original)


In [1]:
# A0 — import seollab
import io, os, pathlib, sys, urllib.request, zipfile

REPO = 'franktome/Dreamerv3_RL_project'
BRANCH = 'kihyun-dreamerv3'
USE_LOCAL = pathlib.Path('seollab/__init__.py').exists()

if USE_LOCAL:
    sys.path.insert(0, str(pathlib.Path('.').resolve()))
    print('LOCAL seollab')
else:
    cache = pathlib.Path.home() / '.cache' / 'seollab_pkg'
    cache.mkdir(parents=True, exist_ok=True)
    url = f'https://github.com/{REPO}/archive/refs/heads/{BRANCH}.zip'
    print('Downloading', url)
    with urllib.request.urlopen(url, timeout=180) as r:
        data = r.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(cache)
    root = next(cache.glob(f'*-{BRANCH}'))
    sys.path.insert(0, str(root))
    print('Installed from GitHub:', root)

from seollab.paths import default_paths
from seollab.env_setup import clone_dreamerv3, setup_jax, ensure_xvfb
from seollab import minecraft, atari, viz, report, inference_demo
from seollab.hf_assets import login_if_needed, ensure_checkpoints, resolve_minecraft_logdir

login_if_needed()  # set HF_TOKEN env var if private assets needed


LOCAL seollab


In [2]:
# A1 — paths & JAX (GPU 1 for viz; highway below uses GPU 0)
import pathlib
from IPython.display import Image, display, Markdown
%matplotlib inline

WORKSPACE = pathlib.Path('.').resolve()
VIZ_GPU = '1'

from seollab.paths import ensure_tmpdir
ensure_tmpdir(WORKSPACE)  # JAX/XLA needs writable temp (not full /tmp)

cfg = default_paths(WORKSPACE, gpu=VIZ_GPU)
clone_dreamerv3(cfg)
setup_jax(cfg)

MC_LOGDIR = resolve_minecraft_logdir(cfg)
RUN_ENV_INFERENCE = (MC_LOGDIR / 'ckpt').exists()
ATARI_GAME = 'pong'
print('Minecraft logdir:', MC_LOGDIR, '| ckpt:', RUN_ENV_INFERENCE)


DreamerV3 found: /mnt/server12_hard0/kiseol/Dreamerv3_RL_project
JAX devices: [CudaDevice(id=0)]
Minecraft logdir: /mnt/server12_hard0/kiseol/Dreamerv3_RL_project/logdir/minecraft_diamond_full | ckpt: True


## A2 — Minecraft DreamerV3 environment rollout

Policy GIF from `logdir/minecraft_diamond_full` checkpoint. DreamerV2: **N/A** for Minecraft.


In [3]:
mc_gif_path = cfg.highlights_dir / 'inference' / 'minecraft_diamond_rollout.gif'
if RUN_ENV_INFERENCE:
    ensure_xvfb(cfg.display)
    mc_result = inference_demo.run_minecraft_env_gif(cfg, logdir=MC_LOGDIR, max_steps=400)
elif mc_gif_path.exists():
    mc_result = {'ok': True, 'gif': str(mc_gif_path)}
else:
    mc_result = {'ok': False, 'error': 'No checkpoint — scores-only mode'}

display(Markdown(
    f"**Minecraft rollout** — steps={mc_result.get('steps', '?')}, "
    f"diamond={mc_result.get('diamond', False)}"
))
if mc_result.get('ok') and pathlib.Path(mc_result.get('gif', '')).exists():
    display(Image(filename=mc_result['gif']))
else:
    display(Markdown(f"*{mc_result.get('error', 'skipped')}*"))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib/python3.11/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


Minecraft action space (25): noop, attack, turn_up, turn_down, turn_left, turn_right, forward, back, left, right, jump, place_dirt, craft_planks, craft_stick, craft_crafting_table, place_crafting_table, craft_wooden_pickaxe, craft_stone_pickaxe, craft_iron_pickaxe, equip_stone_pickaxe, equip_wooden_pickaxe, equip_iron_pickaxe, craft_furnace, place_furnace, smelt_iron_ingot
Observations
  image            Space(uint8, shape=(64, 64, 3), low=0, high=255)
  inventory        Space(float32, shape=(391,), low=0, high=inf)
  inventory_max    Space(float32, shape=(391,), low=0, high=inf)
  equipped         Space(float32, shape=(393,), low=0, high=1)
  reward           Space(float32, shape=(), low=-inf, high=inf)
  health           Space(float32, shape=(), low=-inf, high=inf)
  hunger           Space(float32, shape=(), low=-inf, high=inf)
  breath           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, 

XlaRuntimeError: RESOURCE_EXHAUSTED: /tmp/tempfile-server18-1d6c52d0-1324482-653fa0cef0aaf; No space left on device
	Unable to write PTX contents to: /tmp/tempfile-server18-1d6c52d0-1324482-653fa0cef0aaf

## A3 — Local Minecraft training score distribution


In [ ]:
if (MC_LOGDIR / 'scores.jsonl').exists():
    mc_local = inference_demo.plot_local_minecraft_scores(cfg, logdir=MC_LOGDIR)
    if mc_local.get('ok'):
        display(Image(filename=mc_local['png']))
else:
    display(Markdown('*No local scores.jsonl — skip local distribution.*'))


## A4 — Minecraft official baselines (DreamerV3 vs PPO / IMPALA)


In [ ]:
mc_summary = viz.plot_minecraft_baselines(cfg, show=False)
png = cfg.report_dir / 'minecraft_baselines.png'
display(Image(filename=str(png)))
display(Markdown(f'**Minecraft baselines** — diamond success %: {mc_summary}'))
mc_summary


## A5 — Atari 57: DreamerV2 vs DreamerV3 (official HNS)

Score-based comparison at 50M steps. Live DreamerV2 env rollout omitted (TF/JAX conflict).


In [ ]:
compare = inference_demo.plot_atari_comparison(cfg, show=False)
display(Image(filename=str(cfg.report_dir / 'atari_v2_v3.png')))
display(Image(filename=str(cfg.report_dir / 'inference' / 'atari_hns_distribution.png')))
display(Markdown('**Atari 57** — median HNS V2 vs V3 (official 50M-step scores)'))
compare.sort_values('delta', ascending=False).head(10)


## A6 — Report


In [ ]:
local = mc_result if mc_result.get('ok') else None
report.write_report(cfg, mc_summary, compare, local_mc=local)
display(Markdown(f"Report written to `{cfg.report_dir / 'REPORT.md'}`"))


In [ ]:
# pyvirtualdisplay 없이 바로 렌더링
import gymnasium as gym
import highway_env

env = gym.make('roundabout-v0', render_mode='rgb_array')
print(env.observation_space.shape)
print(env.action_space.n)
obs, _ = env.reset()
frame = env.render()  # 화면 없이 numpy array로 반환
print(frame.shape)    # (H, W, 3) 이면 성공

In [ ]:
# 셀 1 - pyvirtualdisplay 제거, 환경변수만 설정
import sys
import os

import pathlib as _pl
WORKSPACE = _pl.Path('.').resolve()
sys.path.insert(0, str(WORKSPACE))

from seollab.paths import ensure_tmpdir
ensure_tmpdir(WORKSPACE)  # JAX/XLA PTX compile (avoid full /tmp)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

os.environ['JAX_PLATFORMS'] = 'cuda'
# os.environ['LD_LIBRARY_PATH'] = (
#     '/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib/python3.11/'
#     'site-packages/nvidia/cublas/lib:'
#     '/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib/python3.11/'
#     'site-packages/nvidia/cusolver/lib:'
#     '/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib/python3.11/'
#     'site-packages/nvidia/cusparse/lib:'
#     '/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib/python3.11/'
#     'site-packages/nvidia/cufft/lib:'
#     '/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib/python3.11/'
#     'site-packages/nvidia/cudnn/lib:'
#     '/mnt/server12_hard0/kiseol/.conda/envs/dreamerv3/lib'
# )

import jax
print('JAX devices:', jax.devices())

In [ ]:
import elements

# 기존 YAML 읽기 부분을 대체하는 하드코딩된 파이썬 딕셔너리
config_dict = {
    "loss_scales": {
        "rec": 1.0, "rew": 1.0, "con": 1.0, "dyn": 1.0,
        "rep": 0.1, "policy": 1.0, "value": 1.0, "repval": 0.3
    },
    "opt": {
        "lr": 4e-05, "agc": 0.3, "eps": 1e-20, "beta1": 0.9,
        "beta2": 0.999, "momentum": True, "wd": 0.0,
        "schedule": "const", "warmup": 1000, "anneal": 0
    },
    "ac_grads": False,
    "dyn": {
        "typ": "rssm",
        "rssm": {
            "deter": 2048, "hidden": 256, "stoch": 32, "classes": 16,
            "act": "silu", "norm": "rms", "unimix": 0.01,
            "outscale": 1.0, "winit": "trunc_normal_in",
            "imglayers": 2, "obslayers": 1, "dynlayers": 1,
            "absolute": False, "blocks": 8, "free_nats": 1.0
        }
    },
    "enc": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "winit": "trunc_normal_in", "symlog": True,
            "outer": False, "kernel": 5, "strided": False
        }
    },
    "dec": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "outscale": 1.0, "winit": "trunc_normal_in",
            "outer": False, "kernel": 5, "bspace": 8, "strided": False
        }
    },
    "rewhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "conhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "binary", "outscale": 1.0, "winit": "trunc_normal_in"
    },
    "policy": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "minstd": 0.1, "maxstd": 1.0, "outscale": 0.01,
        "unimix": 0.01, "winit": "trunc_normal_in"
    },
    "value": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "policy_dist_disc": "categorical",
    "policy_dist_cont": "bounded_normal",
    "imag_last": 0,
    "imag_length": 15,
    "horizon": 333,
    "contdisc": True,
    "imag_loss": {"slowtar": False, "lam": 0.95, "actent": 0.0003, "slowreg": 1.0},
    "repl_loss": {"slowtar": False, "lam": 0.95, "slowreg": 1.0},
    "slowvalue": {"rate": 0.02, "every": 1},
    "retnorm": {"impl": "perc", "rate": 0.01, "limit": 1.0, "perclo": 5.0, "perchi": 95.0, "debias": False},
    "valnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "advnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "reward_grad": True,
    "repval_loss": True,
    "repval_grad": True,
    "report": True,
    "report_gradnorms": False,
    "logdir": "logdir/highway_roundabout",
    "seed": 0,
    "jax": {
        "platform": "cuda", "compute_dtype": "bfloat16",
        "policy_devices": [0], "train_devices": [0],
        "mock_devices": 0, "prealloc": True, "jit": True,
        "debug": False, "expect_devices": 0, "enable_policy": True,
        "coordinator_address": ""
    },
    "batch_size": 16,
    "batch_length": 64,
    "replay_context": 1,
    "report_length": 32,
    "replica": 0,
    "replicas": 1
}

config =elements.Config(
    config_dict
)

In [ ]:
# 셀 2 - 환경 + 에이전트 로드
import gymnasium as gym
import highway_env
import numpy as np
import imageio
import pathlib
import elements
import ruamel.yaml as yaml
from dreamerv3.agent import Agent
from seollab.hf_assets import ensure_checkpoints

DOWNLOAD_DIR = ensure_checkpoints(pathlib.Path('.'))

obs_space = {
    "obs": elements.Space(np.float32, env.observation_space.shape),
    "reward": elements.Space(np.float32),
    "is_first": elements.Space(bool),
    "is_last": elements.Space(bool),
    "is_terminal": elements.Space(bool),
}
act_space = {"action": elements.Space(np.int32, (), 0, env.action_space.n)}

agent = Agent(obs_space, act_space, config)
cp = elements.Checkpoint(pathlib.Path(DOWNLOAD_DIR) / 'checkpoints/highway_roundabout')
cp.agent = agent
print('ckpt:', list((pathlib.Path(DOWNLOAD_DIR) / 'checkpoints/highway_roundabout').iterdir()))
cp.load()
print('체크포인트 로드 완료!')


In [ ]:
# 셀 3 - 추론 + GIF 저장
env = gym.make('roundabout-v0', render_mode='rgb_array',config={"duration": 100})
frames = []
obs_raw, _ = env.reset()
done = False
total_reward = 0
carry = agent.init_policy(1)
is_first = True

while not done:
    frame = env.render()
    frames.append(frame)

    obs = {
        "obs": np.array([obs_raw], dtype=np.float32),        # (1, 5, 5)
        "reward": np.array([0.0], dtype=np.float32),          # (1,)
        "is_first": np.array([is_first]),                     # (1,)
        "is_last": np.array([False]),                         # (1,)
        "is_terminal": np.array([False]),                     # (1,)
    }
    is_first = False

    carry, act, _ = agent.policy(carry, obs, mode='eval')
    action = int(act['action'][0])
    obs_raw, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    done = terminated or truncated

env.close()
print(f"총 점수: {total_reward:.2f}, 프레임 수: {len(frames)}")

imageio.mimsave('highway_roundabout_inference.gif', frames, fps=10)
print("GIF 저장 완료!")

In [ ]:
# 셀 4 - 노트북에서 바로 보기
from IPython.display import Image
Image('highway_roundabout_inference.gif')